<a href="https://colab.research.google.com/github/jccrews256/ST-554-HW-6/blob/main/Homework_6_Part_I.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 6 Part I

*Class: ST 554*

*Author: Cass Crews*

For Part I of Homework 6, we will continue to query a database that contains information on Major League Baseball.

Before we connect to the database and begin our queries, we need to read in the modules we'll need.

In [7]:
# Reading in required modules
import pandas as pd
import math
import numpy as np
import sqlite3

### Problem 1

We are now ready to connect to the database. We'll utilize the `sqlite3` module to make the connection. After we connect to the dataset, we will query the schema to identify each table in the database.

*Note to reader: The file is too large to be stored in a GitHub repository, so it will need to be brought into your own Colab environment if you are attempting to recreate my results.*

In [8]:
# Make connection to database file; I stored the file "locally" in the colab
# environment
con = sqlite3.connect("lahman_1871-2022.sqlite")

# Constructing SQL query to access tables in schema
get_schema = """
    SELECT *
    FROM sqlite_schema
    WHERE type = "table";
    """

# Obtaining tables from schema
baseball_tables = pd.read_sql(get_schema, con)

# Printing tables
baseball_tables

,type,name,tbl_name,rootpage,sql
0,table,AllstarFull,AllstarFull,2,"CREATE TABLE AllstarFull (\nplayerID TEXT,\nye..."
1,table,Appearances,Appearances,3,"CREATE TABLE Appearances (\nyearID INTEGER,\nt..."
2,table,AwardsManagers,AwardsManagers,4,"CREATE TABLE AwardsManagers (\nplayerID TEXT,\..."
3,table,AwardsPlayers,AwardsPlayers,5,"CREATE TABLE AwardsPlayers (\nplayerID TEXT,\n..."
4,table,AwardsShareManagers,AwardsShareManagers,6,CREATE TABLE AwardsShareManagers (\nawardID TE...
5,table,AwardsSharePlayers,AwardsSharePlayers,7,CREATE TABLE AwardsSharePlayers (\nawardID TEX...
6,table,Batting,Batting,8,"CREATE TABLE Batting (\nplayerID TEXT,\nyearID..."
7,table,BattingPost,BattingPost,9,"CREATE TABLE BattingPost (\nyearID INTEGER,\nr..."
8,table,CollegePlaying,CollegePlaying,10,"CREATE TABLE CollegePlaying (\nplayerID TEXT,\..."
9,table,Fielding,Fielding,11,"CREATE TABLE Fielding (\nplayerID TEXT,\nyearI..."


Note that we have a tremendous amount of information on teams, players, and other key team members (e.g., managers), ranging from ballpark characteristics to information on Hall of Fame members.

### Problem 2

Let's query a few of these tables. To start, we will identify pitchers who are in the Hall of Fame and aggregate a few of their pitching statistics to the career level. This will give us an understanding of what it takes to become a Hall of Fame pitcher. To create this data frame, we will append the regular season (`Pitching`) and postseason (`PitchingPost`) pitching statistics using a `UNION ALL` clause, then join the `HallOfFame` and statistics tables after subsetting to rows in the Hall of Fame table that actually indicate induction rather than simply being up for induction (`inducted = "Y"`). `Pitching` and `PitchingPost` contain each pitcher's pitching statistics for each team they played for in each regular season and postseason, respectively. Thus, we will generate career-level statistics by grouping by `playerID` and summing each statistic across all observations of the appended statistics tables for each pitcher.

To allow for immediate insights, we'll order the pitchers in the data frame by games pitched (`G`). We'll also pull the pitchers' names from the `People` table so we know who these pitchers are.

In [20]:
# Constructing a SQL query to obtain career-level data for each
# pitcher in the HoF
get_pitcher_careers = """
    SELECT FullPitching.playerID, nameLast, nameGiven,
        sum(GS) as GS, sum(G) as G,
        sum(W) as W, sum(L) as L, sum(IPOuts) as IPOuts,
        sum(CG) as CG, sum(SHO) as SHO, sum(SV) as SV
    FROM (SELECT playerID, GS, G, W, L, IPOuts, CG, SHO, SV FROM Pitching
            UNION ALL
            SELECT playerID, GS, G, W, L, IPOuts, CG, SHO, SV FROM PitchingPost)
            FullPitching
    INNER JOIN (SELECT playerID
                FROM HallOfFame
                WHERE inducted = "Y")
                HallOfFame on FullPitching.playerID = HallOfFame.playerID
    LEFT JOIN People on FullPitching.playerID = People.playerID
    GROUP BY FullPitching.playerID, nameLast, nameGiven
    ORDER BY G DESC;
    """

# Reading career-level pitching data into a data frame
hof_pitcher_careers = pd.read_sql(get_pitcher_careers, con)

# Printing the data frame
hof_pitcher_careers

,playerID,nameLast,nameGiven,GS,G,W,L,IPOuts,CG,SHO,SV
0,riverma01,Rivera,Mariano,10,1211,90,61,4274,0,0,694
1,eckerde01,Eckersley,Dennis Lee,362,1099,198,174,9965,100,20,405
2,wilheho01,Wilhelm,James Hoyt,52,1072,143,122,6770,20,5,228
3,hoffmtr01,Hoffman,Trevor William,0,1047,62,77,3307,0,0,605
4,smithle02,Smith,Lee Arthur,6,1026,71,94,3884,0,0,479
...,...,...,...,...,...,...,...,...,...,...,...
103,hoopeha01,Hooper,Harry Bartholomew,0,1,0,0,6,0,0,0
104,kellyge01,Kelly,George Lange,0,1,1,0,15,0,0,0
105,musiast01,Musial,Stanley Frank,0,1,0,0,0,0,0,0
106,speaktr01,Speaker,Tristram Edgar,0,1,0,0,3,0,0,0


We shouldn't be surprised to see Mariano Rivera at the top of the table. He was a legendary closer, and closers play more frequently than starting pitchers.

### Problem 3

For these same Hall of Fame pitchers, let's create a table of career batting statistics. We'll do this by joining the `HallOfFame` table to distinct player ID's in the `Pitching` table to obtain the hall of famers who pitched, then joining that table of pitchers to pre-appended `Batting` and `BattingPost` tables. The `Batting` and `BattingPost` tables hold year-by-team-by-player-level batting statistics for regular seasons and postseasons, respectively. Therefore, we will again use a `GROUP BY` clause to aggregate (sum) the observation-level statistics for each player and obtain career-level metrics.

To accelerate the analysis process, we will again pull in pitcher names. We will also sort by number of home runs, followed by number of hits. The secondary sort is particularly helpful for sorting players with no home runs.

In [21]:
# Constructing a SQL query to obtain career-level batting data for each
# pitcher in the HoF
get_pitcher_batting = """
    SELECT Pitching.playerID, nameLast, nameGiven,
        sum(AB) as AB, sum(R) as R,
        sum(H) as H, sum(HR) as HR, sum(RBI) as RBI,
        sum(BB) as BB, sum(SO) as SO
    FROM (SELECT DISTINCT playerID from Pitching) Pitching
    INNER JOIN (SELECT playerID
                FROM HallOfFame
                WHERE inducted = "Y")
                HallOfFame on Pitching.playerID = HallOfFame.playerID
    LEFT JOIN People on Pitching.playerID = People.playerID
    LEFT JOIN (SELECT playerID, AB, R, H, HR, RBI, BB, SO  FROM Batting
                UNION ALL
                SELECT playerID, AB, R, H, HR, RBI, BB, SO FROM BattingPost)
                FullBatting on Pitching.playerID = FullBatting.playerID
    GROUP BY Pitching.playerID, nameLast, nameGiven
    ORDER BY HR DESC, H DESC;
    """

# Reading career-level batting data into a data frame
hof_pitcher_batting = pd.read_sql(get_pitcher_batting, con)

# Printing the data frame
hof_pitcher_batting

,playerID,nameLast,nameGiven,AB,R,H,HR,RBI,BB,SO
0,ruthba01,Ruth,George Herman,8527,2211,2915,729,2250,2095,1360
1,foxxji01,Foxx,James Emory,8198,1762,2668,538,1933,1461,1321
2,willite01,Williams,Theodore Samuel,7731,1800,2659,521,1840,2026,714
3,musiast01,Musial,Stanley Frank,11058,1958,3652,476,1959,1611,700
4,kellyge01,Kelly,George Lange,6094,830,1803,149,1031,391,717
...,...,...,...,...,...,...,...,...,...,...
103,suttebr01,Sutter,Howard Bruce,103,6,9,0,6,7,50
104,hoffmtr01,Hoffman,Trevor William,34,1,4,0,5,0,11
105,lasorto01,Lasorda,Thomas Charles,14,0,1,0,0,0,4
106,morrija02,Morris,John Scott,5,5,0,0,0,0,3


Many people don't know Babe Ruth was also a pitcher early in his career, but now, reader, you know!

Also, Mariano Rivera may have been a great closer, but he was not a prolific hitter. In fact, he never earned a Major League hit!

### Problem 4

The two data frames we have created so far are rich with information, but it would be helpful to have a single data frame with both pitching and batting statistics. To build such a data frame, we will need to utilize subqueries to handle the fact that not all Hall of Famers who have pitched both batted and pitched each year they played. In particular, we will need to use subqueries to aggregate pitching and batting statistics to the career level before we join the two sets of information.

In [19]:
# Constructing a SQL query to obtain career-level batting and pitching data for
# each pitcher in the HoF
get_pitcher_full_stats = """
    SELECT FullPitching.playerID, nameLast, nameGiven,
        GS, G, W, L, IPOuts, CG, SHO, SV, AB, R, H, HR, RBI, BB, SO
    FROM (SELECT playerID, sum(GS) as GS, sum(G) as G,
            sum(W) as W, sum(L) as L, sum(IPOuts) as IPOuts,
            sum(CG) as CG, sum(SHO) as SHO, sum(SV) as SV
            FROM (SELECT playerID, GS, G, W, L, IPOuts, CG, SHO, SV FROM Pitching
            UNION ALL
            SELECT playerID, GS, G, W, L, IPOuts, CG, SHO, SV FROM PitchingPost)
            FullPitching
        GROUP BY playerID) FullPitching
    INNER JOIN (SELECT playerID
                FROM HallOfFame
                WHERE inducted = "Y")
                HallOfFame on FullPitching.playerID = HallOfFame.playerID
    LEFT JOIN People on FullPitching.playerID = People.playerID
    LEFT JOIN (SELECT playerID, sum(AB) as AB, sum(R) as R,
        sum(H) as H, sum(HR) as HR, sum(RBI) as RBI,
        sum(BB) as BB, sum(SO) as SO
        FROM (SELECT playerID, AB, R, H, HR, RBI, BB, SO  FROM Batting
                UNION ALL
                SELECT playerID, AB, R, H, HR,
                    RBI, BB, SO FROM BattingPost) FullBatting
        GROUP BY playerID) FullBatting on FullPitching.playerID = FullBatting.playerID
    ORDER BY G DESC;
    """

# Reading career-level pitching and batting data into a data frame
hof_pitcher_full_stats = pd.read_sql(get_pitcher_full_stats, con)

# Printing the data frame
hof_pitcher_full_stats

,playerID,nameLast,nameGiven,GS,G,W,L,IPOuts,CG,SHO,SV,AB,R,H,HR,RBI,BB,SO
0,riverma01,Rivera,Mariano,10,1211,90,61,4274,0,0,694,6,0,0,0,1,1,1
1,eckerde01,Eckersley,Dennis Lee,362,1099,198,174,9965,100,20,405,183,9,24,3,12,9,85
2,wilheho01,Wilhelm,James Hoyt,52,1072,143,122,6770,20,5,228,433,24,38,1,21,26,167
3,hoffmtr01,Hoffman,Trevor William,0,1047,62,77,3307,0,0,605,34,1,4,0,5,0,11
4,smithle02,Smith,Lee Arthur,6,1026,71,94,3884,0,0,479,64,2,3,1,2,3,42
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103,hoopeha01,Hooper,Harry Bartholomew,0,1,0,0,6,0,0,0,8877,1442,2493,77,823,1147,503
104,kellyge01,Kelly,George Lange,0,1,1,0,15,0,0,0,6094,830,1803,149,1031,391,717
105,musiast01,Musial,Stanley Frank,0,1,0,0,0,0,0,0,11058,1958,3652,476,1959,1611,700
106,speaktr01,Speaker,Tristram Edgar,0,1,0,0,3,0,0,0,10267,1894,3536,117,1532,1392,327


We can now clearly see that Mariano Rivera was paid millions for his abilities as a closer rather than his batting prowess.

On the opposite end of the spectrum, the great hitter Stan Musial only pitched in a single game.